In [1]:
import numpy as np  #import 模块名 as 简写。大家管numpy叫np，pandas叫pd，matplotlib.pyplot叫plt等。
import torch 
#PyTorch是一个开源的机器学习库，主要用于深度学习应用。它提供了一个灵活的张量计算库和自动求导系统，使得构建和训练神经网络变得更加容易。torch提供了类似于NumPy的张量操作，同时还支持GPU加速，使得大规模数据处理和复杂模型训练更加高效。PyTorch广泛应用于学术研究和工业界，特别是在计算机视觉、自然语言处理和强化学习等领域。
from torch.utils import data #torch.utils.data是PyTorch中用于数据加载和预处理的模块。它提供了DataLoader类，用于批量加载数据，并支持多线程数据加载以提高效率。通过使用DataLoader，用户可以轻松地迭代数据集，进行数据增强和预处理操作，从而简化了训练过程中的数据管理。
from d2l import torch as d2l #从d2l库中导入torch模块，并将其命名为d2l。

true_w = torch.tensor([2, -3.4]) #创建一个包含两个元素的张量true_w，分别为2和-3.4。这通常表示线性回归模型中的权重参数。
true_b = 4.2 #创建一个标量张量true_b，值为4.2。这通常表示线性回归模型中的偏置参数。b是自己定义的变量名，可以是任何合法的标识符，但在机器学习中，b通常用来表示偏置项。偏置项是线性模型中的一个常数项，用于调整模型的输出，使其更好地拟合数据。它允许模型在没有输入特征时也能产生非零输出，从而增加了模型的灵活性和表达能力。b的值可以通过训练数据来学习，以优化模型的性能。但在这里，我们直接将其设置为4.2，作为生成合成数据的参数之一。
features, labels = d2l.synthetic_data(true_w, true_b, 1000) #调用d2l库中的synthetic_data函数，生成一个包含1000个样本的合成数据集。每个样本由特征和标签组成，特征是根据true_w和true_b生成的线性关系加上随机噪声得到的。
#features是一个包含1000个样本的特征张量，每个样本有两个特征，对应于true_w中的两个权重。labels是一个包含1000个标签的张量，每个标签是根据对应的特征和true_w、true_b计算得到的线性关系加上随机噪声得到的。这个数据集可以用于训练线性回归模型，以学习true_w和true_b的值。

#首先

In [2]:
def load_array(data_arrays, batch_size, is_train=True):  #@save   #@save是d2l库中的一个装饰器，用于标记函数或类，它通常用于指示某个函数或类是用户定义的，而不是库中的内置功能。
 #def load_array(data_arrays, batch_size, is_train=True)定义了一个函数load_array，用于构造一个PyTorch数据迭代器。这个函数接受三个参数：data_arrays是一个包含数据数组的元组，batch_size是每个批次的数据量，is_train是一个布尔值，指示是否为训练模式。
    """构造一个PyTorch数据迭代器"""    #数据迭代器允许我们在训练过程中按批次加载数据，通过使用数据迭代器，我们可以轻松地遍历整个数据集，并在每个批次上进行训练或评估。
    dataset = data.TensorDataset(*data_arrays) #创建一个TensorDataset对象dataset，它将data_arrays中的数据数组打包成一个数据集。TensorDataset是PyTorch中用于处理张量数据的类，它将多个张量组合成一个数据集，使得我们可以同时访问多个张量的数据。
    return data.DataLoader(dataset, batch_size, shuffle=is_train) 

batch_size = 10  #在训练过程中，每次迭代将处理10个样本的数据。
data_iter = load_array((features, labels), batch_size) #调用load_array函数，传入特征和标签数据以及批次大小，返回一个数据迭代器data_iter。这个迭代器可以在训练过程中按批次加载数据，使得我们能够高效地处理大规模数据集。

In [3]:
next(iter(data_iter))  #调用iter函数将数据迭代器data_iter转换为一个可迭代对象，然后使用next函数获取下一个批次的数据。这个批次的数据包含特征和标签，通常是一个元组，其中第一个元素是特征张量，第二个元素是标签张量。通过调用next(iter(data_iter))，我们可以查看数据迭代器返回的第一个批次的数据内容。

[tensor([[ 0.0940,  0.2973],
         [ 0.1574, -1.0394],
         [ 1.8693, -0.1212],
         [ 0.6162,  2.5169],
         [ 0.0967,  1.5541],
         [ 0.8098, -0.6710],
         [ 0.5014, -0.0755],
         [ 2.6408, -1.1267],
         [-1.0267,  0.7343],
         [-1.6004, -0.0500]]),
 tensor([[ 3.3804],
         [ 8.0740],
         [ 8.3619],
         [-3.1209],
         [-0.8788],
         [ 8.1012],
         [ 5.4565],
         [13.3288],
         [-0.3623],
         [ 1.1699]])]

In [4]:
# nn是神经网络的缩写
from torch import nn #从torch库中导入nn模块，nn是PyTorch中用于构建神经网络的模块。它提供了各种层、损失函数和优化器等工具，使得构建和训练神经网络变得更加方便和高效。通过使用nn模块，用户可以轻松地定义复杂的神经网络结构，并进行前向传播、反向传播和参数更新等操作，从而实现各种机器学习任务。

net = nn.Sequential(nn.Linear(2, 1))  #创建一个神经网络模型net，使用nn.Sequential类来构建一个顺序容器。这个容器包含一个线性层nn.Linear(2, 1)，表示输入特征的维度为2，输出特征的维度为1。这个模型可以用于线性回归任务，其中输入是两个特征，输出是一个预测值。通过使用nn.Sequential，我们可以方便地将多个层组合在一起，形成一个完整的神经网络模型。

In [5]:
net[0].weight.data.normal_(0, 0.01) #访问net中的第一个层（即线性层）的权重参数，并使用normal_方法将其初始化为均值为0、标准差为0.01的正态分布随机数。这样做可以帮助模型在训练开始时具有较小的权重值，从而有助于模型的收敛和性能提升。
net[0].bias.data.fill_(0)  #访问net中的第一个层（即线性层）的偏置参数，并使用fill_方法将其初始化为0。这样做可以确保模型在训练开始时没有偏置项的影响，从而有助于模型的收敛和性能提升。

tensor([0.])

In [6]:
loss = nn.MSELoss() #创建一个均方误差损失函数对象loss，使用nn.MSELoss类。均方误差（Mean Squared Error）是一种常用的回归损失函数，用于衡量模型预测值与真实值之间的差异。通过使用nn.MSELoss，我们可以在训练过程中计算模型的损失，并通过反向传播来更新模型的参数，从而优化模型的性能。

In [7]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03) #创建一个随机梯度下降优化器对象trainer，使用torch.optim.SGD类。这个优化器将用于更新模型net的参数，以最小化损失函数。参数net.parameters()返回模型的所有可训练参数，lr=0.03指定了学习率，即每次参数更新的步长。通过使用这个优化器，我们可以在训练过程中迭代地调整模型的参数，以提高模型的性能。

In [8]:
num_epochs = 3 #设置训练的轮数，即整个数据集将被迭代多少次。在每个epoch中，模型将遍历整个数据集一次，并更新参数以最小化损失函数。通过增加num_epochs的值，我们可以让模型有更多的机会学习数据中的模式，从而提高模型的性能。然而，过多的epoch可能会导致过拟合，因此需要根据具体情况进行调整。
for epoch in range(num_epochs): #使用for循环来迭代训练过程，循环的次数由num_epochs决定。在每个epoch中，模型将遍历整个数据集一次，并更新参数以最小化损失函数。通过增加num_epochs的值，我们可以让模型有更多的机会学习数据中的模式，从而提高模型的性能。然而，过多的epoch可能会导致过拟合，因此需要根据具体情况进行调整。
    for X, y in data_iter: #在每个epoch中，使用for循环来迭代数据迭代器data_iter，获取每个批次的特征X和标签y。通过遍历数据迭代器，我们可以按批次加载数据，并在每个批次上进行训练或评估。这样做可以提高训练效率，特别是当数据集较大时，可以避免一次性加载整个数据集到内存中。
        l = loss(net(X) ,y) #计算当前批次的损失值l，使用之前定义的损失函数loss。net(X)表示将输入特征X通过模型net进行前向传播，得到模型的预测值。y是当前批次的真实标签。通过计算损失函数，我们可以衡量模型的预测值与真实标签之间的差异，从而指导模型参数的更新以优化性能。
        trainer.zero_grad() #在每次参数更新之前，调用优化器trainer的zero_grad()方法来清除之前计算的梯度。这样做是为了避免梯度累积，因为在PyTorch中，默认情况下，梯度会在每次反向传播时累积。如果不清除之前的梯度，可能会导致参数更新时使用了错误的梯度值，从而影响模型的训练效果。
        l.backward() #调用损失值l的backward()方法来进行反向传播，计算模型参数的梯度。通过反向传播，PyTorch会自动计算出每个参数的梯度值，这些梯度值将用于优化器trainer来更新模型的参数，以最小化损失函数。
        trainer.step() #调用优化器trainer的step()方法来更新模型的参数。这个方法会根据之前计算的梯度值和学习率来调整模型的参数，以最小化损失函数。通过不断地调用step()方法，模型的参数将逐渐优化，从而提高模型的性能。
    l = loss(net(features), labels) #在每个epoch结束后，计算整个数据集上的损失值l。net(features)表示将整个特征数据通过模型net进行前向传播，得到模型的预测值。labels是整个数据集的真实标签。通过计算损失函数，我们可以衡量模型在整个数据集上的性能，从而评估训练的效果。
    print(f'epoch {epoch + 1}, loss {l:f}') #打印当前epoch的编号和对应的损失值l。使用f-string格式化字符串，{epoch + 1}表示当前的epoch编号（从1开始），{l:f}表示损失值l以浮点数格式显示。通过输出这些信息，我们可以跟踪训练过程中的损失变化，从而评估模型的性能提升情况。

epoch 1, loss 0.000231
epoch 2, loss 0.000097
epoch 3, loss 0.000097


In [9]:
w = net[0].weight.data #访问net中的第一个层（即线性层）的权重参数，并将其存储在变量w中。通过访问net[0].weight.data，我们可以获取线性层的权重参数的数值，这些参数是在训练过程中通过优化器trainer进行更新的。通过比较w与true_w，我们可以评估模型学习到的权重参数与真实权重参数之间的差异，从而了解模型的性能。
print('w的估计误差：', true_w - w.reshape(true_w.shape)) #打印权重参数w的估计误差。通过计算true_w与w之间的差异，我们可以评估模型学习到的权重参数与真实权重参数之间的误差。使用reshape方法将w调整为与true_w相同的形状，以便进行元素级别的比较。通过输出这个误差，我们可以了解模型在学习过程中对权重参数的估计情况，从而评估模型的性能。
b = net[0].bias.data #访问net中的第一个层（即线性层）的偏置参数，并将其存储在变量b中。通过访问net[0].bias.data，我们可以获取线性层的偏置参数的数值，这些参数是在训练过程中通过优化器trainer进行更新的。通过比较b与true_b，我们可以评估模型学习到的偏置参数与真实偏置参数之间的差异，从而了解模型的性能。
print('b的估计误差：', true_b - b) #打印偏置参数b的估计误差。通过计算true_b与b之间的差异，我们可以评估模型学习到的偏置参数与真实偏置参数之间的误差。通过输出这个误差，我们可以了解模型在学习过程中对偏置参数的估计情况，从而评估模型的性能。

w的估计误差： tensor([-0.0006,  0.0000])
b的估计误差： tensor([0.0004])
